In [1]:
import json
from tqdm import tqdm

In [4]:
with open('somali-tts-datasets.json') as fopen:
    rows = json.load(fopen)
len(rows)

1719

In [5]:
mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 1719/1719 [00:00<00:00, 3025601.58it/s]


1719

In [8]:
import faiss
import os
import numpy as np

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'somali-tts-datasets/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 1719/1719 [00:00<00:00, 5032.82it/s]


In [9]:
len(data)

1719

In [10]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [12]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'somali-tts-datasets_audio/somali-tts-datasets-data-train-00000-of-00001_0.mp3',
 'text': 'Aamina naftaada wax walba waad awoodaa inaa sameeyso',
 'speaker': 'somali-tts-datasets_audio_0'}

In [13]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'somali-tts-datasets')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 990.16ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 79.2kB / 79.2kB,  217kB/s  
Processing Files (1 / 1): 100%|██████████| 79.2kB / 79.2kB,  198kB/s  
New Data Upload: 100%|██████████| 79.2kB / 79.2kB,  198kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.41 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/2d5c6cb3f280f480afa6329268e55ec756544af7', commit_message='Upload dataset', commit_description='', oid='2d5c6cb3f280f480afa6329268e55ec756544af7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)